# 🇧🇯 Bénin Insights Challenge — Phase 2
## Pipeline Multi-Sources : GDELT × PAG 2021-2026

**Secteurs PAG ciblés :**
- 🏗️ Développement social (infrastructures, capital humain)
- 📊 Gouvernance économique
- 🏥 Santé

**Sources de données :**
| Source | Contenu | Accès |
|--------|---------|-------|
| GDELT | Événements médiatiques | BigQuery (optimisé) |
| Banque Mondiale | PIB, pauvreté, éducation | API gratuite |
| ACLED | Conflits et incidents sécuritaires | CSV téléchargeable |
| OMS/WHO | Indicateurs santé | API gratuite |
| UNICEF | Santé infantile, éducation | CSV téléchargeable |

**Stratégie quota BigQuery :**  
Filtrer d'abord sur `YEAR`, puis `ActionGeo_CountryCode`, puis mots-clés PAG → < 50 GB par extraction complète


## 0. Installation des dépendances

## 1. Configuration générale

In [1]:
import os
import json
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Dossier de sortie des données (structure du dépôt GitHub)
DATA_RAW_DIR = Path("../data/raw")
DATA_PROCESSED_DIR = Path("../data/processed")
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EVENTS_RAW_2021_2026_DATA_PATH = DATA_RAW_DIR / "events_2021_2026.csv"

# Période d'analyse alignée sur le PAG Bénin 2021-2026
YEAR_START = 2021
YEAR_END   = 2026  # Données disponibles dans GDELT jusqu'à 2025

# Code pays Bénin dans GDELT
GDELT_COUNTRY_CODE = 'BN'  # Code FIPS Bénin dans GDELT v2
# Note : tester aussi 'BC' si 'BN' renvoie peu de résultats (les codes varient selon les tables)

print(f"✅ Configuration chargée")
print(f"   Période : {YEAR_START} → {YEAR_END}")
print(f"   Dossier raw      : {DATA_RAW_DIR.resolve()}")
print(f"   Dossier processed: {DATA_PROCESSED_DIR.resolve()}")

✅ Configuration chargée
   Période : 2021 → 2026
   Dossier raw      : E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\data\raw
   Dossier processed: E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\data\processed


In [2]:
# ─── CODES CAMEO LIÉS AUX SECTEURS PAG ───────────────────────────────────────
#
# Les codes CAMEO GDELT classifient les types d'événements géopolitiques.
# On filtre pour ne garder que ceux pertinents au PAG Bénin 2021-2026
#
# Documentation complète : https://www.gdeltproject.org/data/documentation/CAMEO.Manual.1.1b3.pdf

CAMEO_PAG_SECTORS = {
    # Gouvernance économique
    'gouvernance': [
        '036',  # Demander aide économique
        '037',  # Fournir aide économique
        '0341', # Demander aide financière
        '0342', # Fournir aide financière
        '040',  # Consulter (réunions gouvernementales)
        '041',  # Discuter de coopération économique
        '042',  # Accord économique/commercial
        '043',  # Signer accord
    ],
    # Développement social — Infrastructures & Capital humain
    'developpement_social': [
        '071',  # Fournir aide au développement
        '072',  # Fournir aide humanitaire
        '073',  # Fournir aide économique
        '074',  # Fournir aide technique
        '075',  # Fournir aide éducative
        '076',  # Fournir aide médicale
        '0312', # Faire appel pour réforme éducative
        '0314', # Faire appel pour amélioration conditions vie
    ],
    # Santé
    'sante': [
        '076',  # Fournir aide médicale
        '0761', # Fournir aide médicale - urgence
        '0762', # Fournir aide médicale - prévention
        '033',  # Appeler à coopération santé
        '036',  # Demander aide (santé incluse)
    ],
    # Tensions / conflits (pour croiser avec GoldsteinScale)
    'conflits': [
        '14',   # Protester
        '17',   # Actes de violence
        '18',   # Attaques
        '19',   # Combats
        '20',   # Recours à la force non conventionnelle
    ]
}

# Flatten pour le filtre SQL (préfixes = les 2-3 premiers chiffres)
ALL_CAMEO_CODES = []
for codes in CAMEO_PAG_SECTORS.values():
    ALL_CAMEO_CODES.extend(codes)
ALL_CAMEO_CODES = list(set(ALL_CAMEO_CODES))

# Format pour SQL : "'036', '037', ..."
CAMEO_SQL_LIST = ', '.join([f"'{c}'" for c in ALL_CAMEO_CODES])

# Pour les conflits (codes à 2 chiffres = préfixes CAMEO)
CONFLICT_PREFIXES = "'14', '17', '18', '19', '20'"

print(f"✅ {len(ALL_CAMEO_CODES)} codes CAMEO PAG configurés")
print(f"   Secteurs : {list(CAMEO_PAG_SECTORS.keys())}")

✅ 24 codes CAMEO PAG configurés
   Secteurs : ['gouvernance', 'developpement_social', 'sante', 'conflits']


## 1. Nettoyage et préparation du dataset GDELT

In [3]:
# ─── NETTOYAGE ET ENRICHISSEMENT DU DATASET GDELT ────────────────────────────

def clean_gdelt(df: pd.DataFrame) -> pd.DataFrame:
    """
    Nettoie et enrichit le dataset GDELT brut.
    """
    df = df.copy()

    # ── 1. Parsing des dates ──────────────────────────────────────────────────
    df['SQLDATE'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d', errors='coerce')
    df['year']    = df['SQLDATE'].dt.year
    df['month']   = df['SQLDATE'].dt.month
    df['quarter'] = df['SQLDATE'].dt.quarter
    df['year_month'] = df['SQLDATE'].dt.to_period('M').astype(str)

    # ── 2. Nettoyage des valeurs numériques ──────────────────────────────────
    numeric_cols = ['GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes sans GoldsteinScale ni AvgTone (inutilisables pour l'analyse)
    df = df.dropna(subset=['GoldsteinScale', 'AvgTone'])

    # ── 3. Filtrage événements PAG-pertinents ────────────────────────────────

    # Codes CAMEO exacts d'intérêt
    cameo_codes = set(ALL_CAMEO_CODES)

    # Préfixes conflits (2 chiffres)
    conflict_prefixes = tuple(CONFLICT_PREFIXES)

    initial_len = len(df)

    # Sécuriser EventCode en string
    df['EventCode'] = df['EventCode'].astype(str)

    mask_pag = (
        # Codes exacts PAG
        df['EventCode'].isin(cameo_codes)

        # Préfixes conflits
        | df['EventCode'].str[:2].isin(conflict_prefixes)

        # Forte couverture médiatique
        | (df['NumArticles'] >= 5)
    )

    df = df[mask_pag].copy()

    print(f"   Filtrage PAG : {initial_len - len(df):,} événements supprimés")

    # ── 3. Déduplication ─────────────────────────────────────────────────────
    initial_len = len(df)
    df = df.drop_duplicates(subset=['GLOBALEVENTID'])
    print(f"   Déduplication : {initial_len - len(df):,} doublons supprimés")

    # ── 4. Classification par secteur PAG ─────────────────────────────────────
    def classify_pag_sector(event_code: str) -> str:
        if pd.isna(event_code):
            return 'autre'
        code = str(event_code)
        prefix = code[:2]

        # Conflits
        if prefix in ['14', '17', '18', '19', '20']:
            return 'conflit_securite'

        # Santé (aide médicale)
        if code.startswith('076'):
            return 'sante'

        # Développement social
        if code.startswith('07') or code.startswith('031'):
            return 'developpement_social'

        # Gouvernance économique
        if code.startswith('03') or code.startswith('04') or code.startswith('05'):
            return 'gouvernance_economique'

        # Coopération générale
        if code[:1] in ['0', '1', '2']:
            return 'cooperation'

        return 'autre'

    df['secteur_pag'] = df['EventCode'].apply(classify_pag_sector)

    # ── 5. Classification tonalité ────────────────────────────────────────────
    df['tonalite'] = pd.cut(
        df['AvgTone'],
        bins=[-100, -5, -1, 1, 5, 100],
        labels=['tres_negatif', 'negatif', 'neutre', 'positif', 'tres_positif']
    )

    # ── 6. Acteur étranger ou national ───────────────────────────────────────
    df['acteur1_etranger'] = ~df['Actor1CountryCode'].isin(['BEN', 'BJ', ''])
    df['acteur2_etranger'] = ~df['Actor2CountryCode'].isin(['BEN', 'BJ', ''])
    df['implique_acteur_etranger'] = df['acteur1_etranger'] | df['acteur2_etranger']

    # ── 7. Département du Bénin (depuis ADM1Code) ────────────────────────────
    dept_mapping = {
        'BN01': 'Alibori',    'BN02': 'Atacora',    'BN03': 'Atlantique',
        'BN04': 'Borgou',     'BN05': 'Collines',   'BN06': 'Couffo',
        'BN07': 'Donga',      'BN08': 'Littoral',   'BN09': 'Mono',
        'BN10': 'Oueme',      'BN11': 'Plateau',    'BN12': 'Zou',
    }
    df['departement'] = df['ActionGeo_ADM1Code'].map(dept_mapping).fillna('Inconnu')

    print(f"✅ GDELT nettoyé : {len(df):,} événements")
    print(f"   Secteurs PAG : {df['secteur_pag'].value_counts().to_dict()}")
    return df


In [5]:
# ── EXÉCUTER ──────────────────────────────────────────────────────────────────
chunks = []
for chunk in pd.read_csv(EVENTS_RAW_2021_2026_DATA_PATH, chunksize=100_000):
    chunk_clean = clean_gdelt(chunk)
    chunks.append(chunk_clean)
gdelt_clean = pd.concat(chunks, ignore_index=True)
gdelt_clean.head()
gdelt_clean.to_csv(DATA_PROCESSED_DIR / 'gdelt_2021_2026_clean.csv', index=False)


   Filtrage PAG : 50,854 événements supprimés
   Déduplication : 0 doublons supprimés
✅ GDELT nettoyé : 49,144 événements
   Secteurs PAG : {'autre': 25524, 'conflit_securite': 12997, 'cooperation': 10623}
   Filtrage PAG : 32,658 événements supprimés
   Déduplication : 0 doublons supprimés
✅ GDELT nettoyé : 27,106 événements
   Secteurs PAG : {'autre': 13584, 'conflit_securite': 8234, 'cooperation': 5288}


## 2. Fusion multi-sources (dataset unifié mensuel)

In [6]:
# ─── AGRÉGATION GDELT PAR MOIS ET SECTEUR ────────────────────────────────────

def aggregate_gdelt_monthly(df_gdelt: pd.DataFrame) -> pd.DataFrame:
    """
    Agrège les événements GDELT par mois et secteur PAG.
    Produit les variables clés pour la jointure avec les sources externes.
    """
    df = df_gdelt.copy()

    agg = df.groupby(['year', 'month', 'secteur_pag']).agg(
        nb_evenements         = ('GLOBALEVENTID', 'count'),
        ton_moyen             = ('AvgTone', 'mean'),
        goldstein_moyen       = ('GoldsteinScale', 'mean'),
        total_mentions        = ('NumMentions', 'sum'),
        total_articles        = ('NumArticles', 'sum'),
        pct_acteurs_etrangers = ('implique_acteur_etranger', 'mean'),
        ton_std               = ('AvgTone', 'std'),       # Volatilité médiatique
        nb_sources_distinctes = ('NumSources', 'sum'),
    ).reset_index()

    agg['ton_moyen']             = agg['ton_moyen'].round(3)
    agg['goldstein_moyen']       = agg['goldstein_moyen'].round(3)
    agg['pct_acteurs_etrangers'] = (agg['pct_acteurs_etrangers'] * 100).round(1)

    print(f"✅ Agrégation mensuelle GDELT : {len(agg):,} lignes (année × mois × secteur)")
    return agg

def build_unified_dataset(
    df_gdelt_monthly: pd.DataFrame,
    df_worldbank: pd.DataFrame,
    df_acled_monthly: pd.DataFrame = None,
    df_who: pd.DataFrame = None,
    cache_path: Path = None
) -> pd.DataFrame:
    """
    Fusionne toutes les sources en un dataset unifié.

    Clé de jointure :
    - GDELT + ACLED → par (year, month)
    - Banque Mondiale + OMS → par (year) → données annuelles broadcastées sur les mois
    """
    print("🔗 Fusion multi-sources en cours...")
    df = df_gdelt_monthly.copy()

    # Jointure avec les données annuelles BM (broadcast year → months)
    if df_worldbank is not None and not df_worldbank.empty:
        df = df.merge(df_worldbank, on='year', how='left')
        print(f"   ✅ Banque Mondiale fusionnée ({len(df_worldbank.columns)-1} indicateurs)")

    # Jointure avec OMS (annuel)
    if df_who is not None and not df_who.empty:
        who_cols = [c for c in df_who.columns if c != 'year']
        df = df.merge(df_who[['year'] + who_cols], on='year', how='left')
        print(f"   ✅ OMS fusionné ({len(who_cols)} indicateurs santé)")

    # Jointure avec ACLED (mensuel)
    if df_acled_monthly is not None and not df_acled_monthly.empty:
        df = df.merge(df_acled_monthly, on=['year', 'month'], how='left')
        df['acled_nb_incidents']  = df['acled_nb_incidents'].fillna(0)
        df['acled_nb_fatalities'] = df['acled_nb_fatalities'].fillna(0)
        print(f"   ✅ ACLED fusionné")

    # Trier
    df = df.sort_values(['year', 'month', 'secteur_pag']).reset_index(drop=True)

    # Sauvegarder
    if cache_path:
        df.to_csv(cache_path, index=False)
        print(f"   💾 Dataset unifié sauvegardé : {cache_path}")

    print(f"\n✅ Dataset unifié : {len(df):,} lignes × {len(df.columns)} colonnes")
    print(f"   Colonnes : {list(df.columns)}")
    return df


# ── EXÉCUTER ──────────────────────────────────────────────────────────────────
# Décommenter une fois toutes les sources chargées

# df_gdelt_monthly = aggregate_gdelt_monthly(df_gdelt_clean)
# df_acled_monthly = aggregate_acled_monthly(df_acled) if 'df_acled' in dir() else pd.DataFrame()

# 
print("✅ Fonctions de fusion prêtes")

✅ Fonctions de fusion prêtes


In [7]:
df_gdelt_clean = pd.read_csv(DATA_PROCESSED_DIR / 'gdelt_2021_2026_clean.csv')
df_worldbank = pd.read_csv(DATA_RAW_DIR / 'worldbank_benin_2021_2025.csv')
df_who       = pd.read_csv(DATA_RAW_DIR / 'who_benin_2021_2025.csv')

df_gdelt_monthly = aggregate_gdelt_monthly(df_gdelt_clean)
df_unified = build_unified_dataset(
     df_gdelt_monthly = df_gdelt_monthly,
     df_worldbank     = df_worldbank,
     df_who           = df_who,
     cache_path       = DATA_PROCESSED_DIR / 'dataset_unifie_benin.csv'
)


✅ Agrégation mensuelle GDELT : 195 lignes (année × mois × secteur)
🔗 Fusion multi-sources en cours...
   ✅ Banque Mondiale fusionnée (14 indicateurs)
   ✅ OMS fusionné (5 indicateurs santé)
   💾 Dataset unifié sauvegardé : ..\data\processed\dataset_unifie_benin.csv

✅ Dataset unifié : 195 lignes × 30 colonnes
   Colonnes : ['year', 'month', 'secteur_pag', 'nb_evenements', 'ton_moyen', 'goldstein_moyen', 'total_mentions', 'total_articles', 'pct_acteurs_etrangers', 'ton_std', 'nb_sources_distinctes', 'investissement_brut_pct_pib', 'inflation_pct', 'ide_entrants_pct_pib', 'acces_electricite_pct', 'acces_eau_potable_pct', 'utilisateurs_internet_pct', 'pauvrete_extreme_pct', 'taux_alphabetisation_adultes', 'mortalite_infantile_pour_1000', 'esperance_vie_ans', 'lits_hopitaux_pour_1000', 'depenses_sante_pct_pib', 'mortalite_maternelle_pour_100000', 'vaccination_rougeole_pct', 'esperance_vie_naissance', 'mortalite_maternelle', 'mortalite_enfants_moins5', 'couverture_antenatal_4visites_pct', 'a